# MobRecon — Live Hand Mesh Demo

Real-time 3D hand mesh reconstruction using [MobRecon / HandMesh](https://github.com/SeanChenxy/HandMesh) on the Mac front-facing camera.

**Press Q or Ctrl+C to close the demo window.**

---

Run cells **in order**:
1. Install dependencies
2. Clone HandMesh repo
3. Download model weights  
4. Build model
5. Launch camera demo

In [6]:
# ── Cell 1: Install Dependencies ─────────────────────────────────────────────
import subprocess, sys

def pip(*pkgs):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *pkgs])

print('Installing core packages...')
pip('torch', 'torchvision')
pip('opencv-python')
pip('mediapipe')
pip('fvcore')
pip('trimesh')   # replaces openmesh (no binary wheel on Python 3.8/Mac)
pip('gdown', 'yacs', 'scipy', 'plyfile', 'chumpy')

# torch-scatter / torch-sparse needed by MobRecon spiral convolutions
import torch
v = torch.__version__.split('+')[0]
pyg_whl = f'https://data.pyg.org/whl/torch-{v}+cpu.html'
print(f'Installing torch-geometric wheels for PyTorch {v} (CPU)...')
try:
    pip('torch-scatter', 'torch-sparse', '-f', pyg_whl)
    pip('torch-geometric')
    print('torch-geometric OK')
except Exception as e:
    print(f'torch-geometric install failed: {e}')
    print('If this fails, see: https://pytorch-geometric.readthedocs.io/en/latest/install/installation.html')

print('\nAll dependencies ready.')

Installing core packages...


You should consider upgrading via the '/Users/christianluisefendy/.pyenv/versions/3.8.10/bin/python -m pip install --upgrade pip' command.
You should consider upgrading via the '/Users/christianluisefendy/.pyenv/versions/3.8.10/bin/python -m pip install --upgrade pip' command.
You should consider upgrading via the '/Users/christianluisefendy/.pyenv/versions/3.8.10/bin/python -m pip install --upgrade pip' command.
You should consider upgrading via the '/Users/christianluisefendy/.pyenv/versions/3.8.10/bin/python -m pip install --upgrade pip' command.
You should consider upgrading via the '/Users/christianluisefendy/.pyenv/versions/3.8.10/bin/python -m pip install --upgrade pip' command.
You should consider upgrading via the '/Users/christianluisefendy/.pyenv/versions/3.8.10/bin/python -m pip install --upgrade pip' command.


Installing torch-geometric wheels for PyTorch 2.4.1 (CPU)...


You should consider upgrading via the '/Users/christianluisefendy/.pyenv/versions/3.8.10/bin/python -m pip install --upgrade pip' command.


torch-geometric OK

All dependencies ready.


You should consider upgrading via the '/Users/christianluisefendy/.pyenv/versions/3.8.10/bin/python -m pip install --upgrade pip' command.


In [7]:
# ── Cell 2: Clone HandMesh Repository ────────────────────────────────────────
import subprocess, sys, os

REPO_DIR = os.path.join(os.getcwd(), 'HandMesh')

if not os.path.exists(REPO_DIR):
    print('Cloning HandMesh (shallow)...')
    subprocess.check_call([
        'git', 'clone', '--depth', '1',
        'https://github.com/SeanChenxy/HandMesh.git',
        REPO_DIR
    ])
    print('Cloned.')
else:
    print('HandMesh already present.')

# Add repo root so Python can find: mobrecon.*, conv.*, utils.*
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

print(f'REPO_DIR = {REPO_DIR}')

HandMesh already present.
REPO_DIR = /Users/christianluisefendy/Documents/AIML_Challenge1_Vision/HandMesh


In [8]:
# ── Cell 3: Download Model Weights ───────────────────────────────────────────
import gdown, os, glob

CKPT_DIR   = os.path.join(REPO_DIR, 'mobrecon', 'checkpoints')
GDRIVE_URL = 'https://drive.google.com/drive/folders/1MIE0Jo01blG6RWo2trQbXlQ92tMOaLx_'
os.makedirs(CKPT_DIR, exist_ok=True)

def find_checkpoints(directory):
    return (glob.glob(os.path.join(directory, '**', '*.pt'),  recursive=True) +
            glob.glob(os.path.join(directory, '**', '*.pth'), recursive=True))

existing = find_checkpoints(CKPT_DIR)

if not existing:
    print(f'Downloading checkpoints from Google Drive to {CKPT_DIR} ...')
    print('(May take several minutes — folder contains multiple model checkpoints.)')
    try:
        gdown.download_folder(
            url=GDRIVE_URL,
            output=CKPT_DIR,
            quiet=False,
            remaining_ok=True,
        )
        existing = find_checkpoints(CKPT_DIR)
    except Exception as e:
        print(f'\nAuto-download error: {e}')
        print('\nManual fallback:')
        print(f'  1. Visit: {GDRIVE_URL}')
        print(f'  2. Download a MobRecon *.pt checkpoint into: {CKPT_DIR}')
        print('  Then re-run this cell.')

if not existing:
    raise FileNotFoundError(
        f'No checkpoint (.pt/.pth) found under {CKPT_DIR}.\n'
        f'Download weights from: {GDRIVE_URL}'
    )

# Prefer a checkpoint with 'mobrecon' in the filename
mobrecon_ckpts = [f for f in existing if 'mobrecon' in os.path.basename(f).lower()]
CKPT_PATH = (mobrecon_ckpts or existing)[0]
print(f'\nUsing checkpoint: {CKPT_PATH}')


Using checkpoint: /Users/christianluisefendy/Documents/AIML_Challenge1_Vision/HandMesh/mobrecon/checkpoints/mobrecon_densestack.pt


In [ ]:
# ── Cell 4: Build MobRecon Model ─────────────────────────────────────────────
# 1) Inject lightweight shims for openmesh & psbody.mesh BEFORE any HandMesh
#    code imports them. No arm64 macOS wheel exists for openmesh; psbody.mesh
#    is only used to regenerate transform.pkl from scratch (we use the cached
#    one shipped in the repo).
import sys, types
import numpy as _np_shim


class _VertexHandle:
    __slots__ = ('_idx',)
    def __init__(self, idx):
        self._idx = int(idx)
    def idx(self):
        return self._idx


class _TriMesh:
    """Pure-Python replacement for openmesh.TriMesh covering the API
    that HandMesh's spiral computation needs."""
    def __init__(self, vertices, faces):
        self._points = _np_shim.asarray(vertices, dtype=_np_shim.float64).copy()
        self._faces  = _np_shim.asarray(faces, dtype=_np_shim.int64).copy()
        self._build_topology()

    def _build_topology(self):
        n_v = len(self._points)
        v_face_pos = [[] for _ in range(n_v)]
        for fi, face in enumerate(self._faces):
            for i in range(3):
                v_face_pos[int(face[i])].append((fi, i))
        edge_to_fp = {}
        for fi, face in enumerate(self._faces):
            for i in range(3):
                edge_to_fp[(int(face[i]), int(face[(i+1) % 3]))] = (fi, i)
        next_fp, prev_fp = {}, {}
        for fi, face in enumerate(self._faces):
            for i in range(3):
                v_, u_ = int(face[i]), int(face[(i+2) % 3])  # incoming source
                key = (v_, u_)
                if key in edge_to_fp and edge_to_fp[key] != (fi, i):
                    next_fp[(fi, i)] = edge_to_fp[key]
                    prev_fp[edge_to_fp[key]] = (fi, i)
        self._neighbors = [[] for _ in range(n_v)]
        for v_ in range(n_v):
            fps = v_face_pos[v_]
            if not fps:
                continue
            start = next((fp for fp in fps if fp not in prev_fp), fps[0])
            walk, cur, seen = [], start, set()
            while cur is not None and cur not in seen:
                seen.add(cur)
                walk.append(cur)
                cur = next_fp.get(cur)
            res = [int(self._faces[fi][(i+1) % 3]) for (fi, i) in walk]
            if cur is None and walk:
                last_fi, last_i = walk[-1]
                a_ = int(self._faces[last_fi][(last_i+2) % 3])
                if a_ not in res:
                    res.append(a_)
            self._neighbors[v_] = res

    def vertices(self):
        for i in range(len(self._points)):
            yield _VertexHandle(i)

    def vv(self, vh):
        v = vh.idx() if isinstance(vh, _VertexHandle) else int(vh)
        for n in self._neighbors[v]:
            yield _VertexHandle(n)

    def points(self):
        return self._points


# Install openmesh shim if real openmesh is unavailable
if 'openmesh' not in sys.modules:
    try:
        import openmesh  # noqa: F401
    except ImportError:
        _shim = types.ModuleType('openmesh')
        _shim.VertexHandle = _VertexHandle
        _shim.TriMesh      = _TriMesh
        _shim.read_trimesh = lambda path: None  # not used (utils.read.read_mesh uses trimesh)
        _shim.write_mesh   = lambda *a, **k: None
        sys.modules['openmesh'] = _shim
        print('Using openmesh shim (no native openmesh installed).')

# Install psbody.mesh stub (mesh_sampling imports it at top level but we never call it)
if 'psbody' not in sys.modules:
    try:
        import psbody.mesh  # noqa: F401
    except ImportError:
        _ps     = types.ModuleType('psbody')
        _ps_msh = types.ModuleType('psbody.mesh')
        class _StubMesh:
            def __init__(self, *a, **k):
                raise RuntimeError(
                    'psbody.mesh.Mesh stub called — install MPI-IS mesh '
                    'only if you need to regenerate transform.pkl from scratch.'
                )
        _ps_msh.Mesh = _StubMesh
        _ps.mesh     = _ps_msh
        sys.modules['psbody']      = _ps
        sys.modules['psbody.mesh'] = _ps_msh
        print('Using psbody.mesh stub.')

# 2) Now safe to import HandMesh model code ────────────────────────────────
import torch, os, numpy as np
import fvcore
from mobrecon.configs.config import get_cfg
from mobrecon.models.mobrecon_ds import MobRecon_DS

# Build config with defaults (template paths resolve relative to config file location)
cfg = get_cfg()

print('Building MobRecon_DS...')
model = MobRecon_DS(cfg)

# Load checkpoint — handle several common serialisation formats
print(f'Loading weights from:\n  {CKPT_PATH}')
raw = torch.load(CKPT_PATH, map_location='cpu')
if isinstance(raw, dict):
    state = (raw.get('model_state_dict') or
             raw.get('state_dict')       or
             raw.get('model')            or
             raw)
else:
    state = raw
missing, unexpected = model.load_state_dict(state, strict=False)
if missing:
    print(f'  Missing keys  : {len(missing)}')
if unexpected:
    print(f'  Unexpected keys: {len(unexpected)}')

# Device — MPS for Apple Silicon, CUDA, or CPU
DEVICE = ('mps'  if torch.backends.mps.is_available()  else
          'cuda' if torch.cuda.is_available()           else
          'cpu')
model.to(DEVICE).eval()

# Face topology for mesh wireframe (shape: N_faces × 3)
FACES = np.load(os.path.join(REPO_DIR, 'template', 'right_faces.npy')).astype(np.int32)

# Standard MANO 21-joint skeleton edges
HAND_SKELETON = [
    (0,1),(1,2),(2,3),(3,4),          # thumb
    (0,5),(5,6),(6,7),(7,8),          # index
    (0,9),(9,10),(10,11),(11,12),     # middle
    (0,13),(13,14),(14,15),(15,16),   # ring
    (0,17),(17,18),(18,19),(19,20),   # pinky
    (5,9),(9,13),(13,17),             # metacarpal arch
]

print(f'\nModel ready  |  device: {DEVICE}  |  faces: {len(FACES)}')

/Users/christianluisefendy/Documents/AIML_Challenge1_Vision/HandMesh/mobrecon/models/densestack.py:234: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  weight = torch.load(os.

Building MobRecon_DS...
Load pre-trained weight: densestack.pth


TypeError: int() argument must be a string, a bytes-like object or a number, not '_VertexHandle'

: 

In [ ]:
# ── Cell 5: Live Camera Demo — Ctrl+C or Q to quit ───────────────────────────
import cv2, signal, numpy as np, torch
import mediapipe as mp

INPUT_SIZE = 128
IMG_MEAN   = 0.5          # mobrecon normalises with mean=0.5, std=0.5
IMG_STD    = 0.5
BBOX_PAD   = 0.35         # pad bbox by this fraction of hand extent
MESH_COLOR = (0, 220, 50) # green wireframe
SKEL_COLOR = (30, 100, 255)
KPT_COLOR  = (0, 255, 255)

# ── MediaPipe hand detector ────────────────────────────────────────────────────
mp_hands     = mp.solutions.hands
hand_tracker = mp_hands.Hands(
    static_image_mode=False,
    max_num_hands=1,
    min_detection_confidence=0.7,
    min_tracking_confidence=0.5,
)

# ── Helper functions ──────────────────────────────────────────────────────────

def get_bbox(mp_res, H, W, pad=BBOX_PAD):
    """Return (x1,y1,x2,y2) padded hand bbox from MediaPipe result, or None."""
    if not mp_res.multi_hand_landmarks:
        return None
    lms = mp_res.multi_hand_landmarks[0].landmark
    xs  = [lm.x * W for lm in lms]
    ys  = [lm.y * H for lm in lms]
    bw, bh = max(xs) - min(xs), max(ys) - min(ys)
    x1 = max(0, int(min(xs) - bw * pad))
    y1 = max(0, int(min(ys) - bh * pad))
    x2 = min(W, int(max(xs) + bw * pad))
    y2 = min(H, int(max(ys) + bh * pad))
    return (x1, y1, x2, y2)


def preprocess(frame_rgb, bbox):
    """Crop hand, resize to 128×128, normalise → (1,3,128,128) tensor."""
    x1, y1, x2, y2 = bbox
    crop = frame_rgb[y1:y2, x1:x2]
    if crop.size == 0:
        return None
    crop = cv2.resize(crop, (INPUT_SIZE, INPUT_SIZE))
    crop = (crop.astype(np.float32) / 255.0 - IMG_MEAN) / IMG_STD
    return torch.from_numpy(crop).permute(2, 0, 1).unsqueeze(0).float().to(DEVICE)


def joints_to_px(joint_img, bbox):
    """
    joint_img (21,2): model output — may be [0,1], [-1,1], or pixel [0,128].
    Returns integer pixel coords in the full frame.
    """
    j = joint_img.copy().astype(np.float32)
    # Detect coordinate space and normalise to [0,1]
    if j.min() < -0.5:          # [-1, 1] space
        j = (j + 1.0) / 2.0
    elif j.max() > 1.5:         # pixel space [0, INPUT_SIZE]
        j = j / float(INPUT_SIZE)
    x1, y1, x2, y2 = bbox
    out = np.stack([
        j[:, 0] * (x2 - x1) + x1,
        j[:, 1] * (y2 - y1) + y1,
    ], axis=1).astype(np.int32)
    return out


def verts_to_px(verts3d, joints_px):
    """
    Project verts XY to image space by aligning their 2D centroid and
    extent to the predicted joint locations.
    """
    j_cx  = float(joints_px[:, 0].mean())
    j_cy  = float(joints_px[:, 1].mean())
    j_ext = float(max(joints_px[:, 0].ptp(), joints_px[:, 1].ptp())) + 1e-6
    v_cx  = float(verts3d[:, 0].mean())
    v_cy  = float(verts3d[:, 1].mean())
    v_ext = float(max(verts3d[:, 0].ptp(), verts3d[:, 1].ptp())) + 1e-6
    s = j_ext / v_ext
    return np.stack([
        (verts3d[:, 0] - v_cx) * s + j_cx,
        (verts3d[:, 1] - v_cy) * s + j_cy,
    ], axis=1).astype(np.int32)


def draw_skeleton(img, pts, edges, color, thickness=2):
    for a, b in edges:
        cv2.line(img, tuple(pts[a]), tuple(pts[b]), color, thickness)


def draw_wireframe(img, verts_px, faces, color, thickness=1):
    H, W = img.shape[:2]
    for f in faces:
        p = verts_px[f]   # (3, 2)
        # Skip any triangle with a vertex outside the frame
        if (p < 0).any() or (p[:, 0] >= W).any() or (p[:, 1] >= H).any():
            continue
        cv2.line(img, tuple(p[0]), tuple(p[1]), color, thickness)
        cv2.line(img, tuple(p[1]), tuple(p[2]), color, thickness)
        cv2.line(img, tuple(p[2]), tuple(p[0]), color, thickness)


# ── Signal / keyboard handling ────────────────────────────────────────────────
running = True

def _stop(sig, frame_):
    global running
    running = False

signal.signal(signal.SIGINT, _stop)

# ── Open Mac front camera ─────────────────────────────────────────────────────
cap = cv2.VideoCapture(0, cv2.CAP_AVFOUNDATION)
if not cap.isOpened():
    cap = cv2.VideoCapture(0)   # fallback without AVFoundation flag
assert cap.isOpened(), 'Could not open camera. Check System Settings → Privacy → Camera.'

cap.set(cv2.CAP_PROP_FRAME_WIDTH,  1280)
cap.set(cv2.CAP_PROP_FRAME_HEIGHT,  720)

print('Demo running — hold a hand in front of the camera.')
print('Press Q in the window or Ctrl+C here to quit.\n')

try:
    while running:
        ok, frame = cap.read()
        if not ok:
            break

        # Mirror so the display feels natural (like a mirror)
        frame     = cv2.flip(frame, 1)
        frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        H, W      = frame.shape[:2]

        mp_res = hand_tracker.process(frame_rgb)
        bbox   = get_bbox(mp_res, H, W)

        if bbox is not None:
            tensor = preprocess(frame_rgb, bbox)
            if tensor is not None:
                with torch.no_grad():
                    out = model(tensor)

                # Model outputs
                joint_img = out['joint_img'][0].cpu().numpy()  # (21, 2)
                verts_3d  = out['verts'][0].cpu().numpy()      # (778, 3)

                # Map to image pixel coordinates
                jpts = joints_to_px(joint_img, bbox)
                vpts = verts_to_px(verts_3d, jpts)

                # Draw: mesh first (background), then skeleton, then keypoints
                draw_wireframe(frame, vpts, FACES, MESH_COLOR)
                draw_skeleton(frame,  jpts, HAND_SKELETON, SKEL_COLOR)
                for pt in jpts:
                    cv2.circle(frame, tuple(pt), 4, KPT_COLOR, -1)

            # Bounding box
            x1, y1, x2, y2 = bbox
            cv2.rectangle(frame, (x1, y1), (x2, y2), (180, 180, 180), 1)

        cv2.putText(frame, 'MobRecon  |  Q to quit',
                    (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 255, 255), 2)
        cv2.imshow('MobRecon Demo', frame)

        if cv2.waitKey(1) & 0xFF == ord('q'):
            break

except KeyboardInterrupt:
    pass

finally:
    cap.release()
    cv2.destroyAllWindows()
    hand_tracker.close()
    print('Demo closed.')

INFO: Created TensorFlow Lite XNNPACK delegate for CPU.


Demo running — hold a hand in front of the camera.
Press Q in the window or Ctrl+C here to quit.

Demo closed.


NameError: name 'DEVICE' is not defined

: 